# Итоговое решение: прогноз GMV

Ноутбук собирает выбранный итоговый сабмит `submission_monetary_cadence_inactive_guard.csv` из сохранённых прогнозов. Для пользователей без активных дней заказов берётся контрольный прогноз; для остальных — смесь контрольной модели и модели денежного ритма.

Полное переобучение запускается модулями из `src/`, потому что требует много памяти.

## 1. Окружение

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)
print('Python:', sys.executable)

Project root: /Users/danasokol/Desktop/ML - соревы/OZON_GMV
Python: /Users/danasokol/.venvs/ozon-ecup/bin/python


## 2. Сборка итогового CSV

Ячейка повторно проверяет правило на двух валидационных частях, применяет его к 250 000 тестовых пользователей и сохраняет сабмит.

In [2]:
from src.inactive_guard import SUBMISSION_PATH, build_submission

final_summary = build_submission()
final_summary

{'experiment': 'inactive_order_guard_v1',
 'rule': 'candidate_weight=0 if order_active_days=0 else 0.4',
 'rule_fixed_on': 'tune',
 'tune_gain_vs_global_blend': 7.583283828882514e-06,
 'confirm_gain_vs_global_blend': 4.915968066598886e-05,
 'confirm_bootstrap_ci_low': 1.3968277905601623e-05,
 'confirm_bootstrap_ci_high': 8.630509605520029e-05,
 'confirm_bootstrap_positive_share': 0.998,
 'accepted': True,
 'submission_path': '/Users/danasokol/Desktop/ML - соревы/OZON_GMV/submissions/submission_monetary_cadence_inactive_guard.csv',
 'rows': 250000,
 'inactive_users': 31971,
 'inactive_share': 0.127884,
 'prediction_sum': 9455203.970805557,
 'prediction_mean': 37.82081588322223,
 'prediction_median': 6.920855942194702,
 'zero_share': 0.001376}

## 3. Проверка файла

In [3]:
submission = pd.read_csv(SUBMISSION_PATH)
assert submission.columns.tolist() == ['user_id', 'predict']
assert len(submission) == 250_000
assert submission['user_id'].is_unique
assert np.isfinite(submission['predict']).all()
assert (submission['predict'] >= 0).all()

validation_result = {
    'path': str(SUBMISSION_PATH),
    'rows': len(submission),
    'unique_users': int(submission['user_id'].nunique()),
    'prediction_sum': float(submission['predict'].sum()),
    'prediction_mean': float(submission['predict'].mean()),
}
validation_result

{'path': '/Users/danasokol/Desktop/ML - соревы/OZON_GMV/submissions/submission_monetary_cadence_inactive_guard.csv',
 'rows': 250000,
 'unique_users': 250000,
 'prediction_sum': 9455203.970805557,
 'prediction_mean': 37.82081588322223}

## 4. Полное переобучение

Полная последовательность из исходного `data/train.parquet`:

1. `python -m src.final_solution --retrain` — базовая годовая/ритмическая модель.
2. Вызвать `prepare_experiment_inputs()`, `run_validation_experiment()` и `run_final_training()` из `src.monetary_experiment` — модель денежного ритма.
3. `python -m src.inactive_guard` — итоговое правило для неактивных пользователей.